<style>
ul { padding-left:24px; }
table { margin-left:24px !important; margin-right:0 !important; }
th, td { text-align:left !important; }
li table { margin-left:0 !important; }
</style>

# 생명과학 예제: 생장 조건

## 1) 예제 설명

- 조명과 양액(nutrient solution)의 단계를 골라 식물의 생장량 최대화
- 같은 품종과 재배 기간에서 가상 생장량 비교, 단위는 mm

| 조명 / 양액 | 낮음 (0) | 중간 (1) | 높음 (2) |
|:---|:---|:---|:---|
| 낮음 (0) | 3 | 5 | 4 |
| 중간 (1) | 6 | 9 | 7 |
| 높음 (2) | 5 | 8 | 6 |

- 의사결정벡터(decision vector): $\mathbf{x}=(x_1,x_2)$, 두 성분은 조명과 양액 단계, 각 단계는 0·1·2
- 입자 위치: 탐색에 사용하는 실수 벡터 $\mathbf{z}=(z_1,z_2)$, 각 성분의 범위는 0~2
- 목적함수(objective function): 선택한 행과 열의 생장량, 큰 값이 좋은 결과

- `[1, 2]`는 조명 중간·양액 높음, 표에서 생장량 7 mm에 해당


<style>
ul { padding-left:24px; }
table { margin-left:24px !important; margin-right:0 !important; }
th, td { text-align:left !important; }
li table { margin-left:0 !important; }
</style>

## 2) 구현

- 공통 PSO 함수는 탐색 후 전역 최적 위치 gbest 반환

- 초기 속도의 각 성분은 해당 탐색 구간 폭의 −10%~10%에서 균등분포 난수로 설정


In [ ]:
import numpy as np


def particle_swarm(initial_positions, objective, lower, upper, iterations=20, seed=0):
    # 난수 생성기와 초기 위치 설정
    rng = np.random.default_rng(seed)
    positions = np.array(initial_positions, dtype=float, copy=True)
    if np.any(positions < lower) or np.any(positions > upper):
        raise ValueError("Initial positions must satisfy the bounds.")

    # 탐색 구간 폭에 비례하는 초기 속도 설정
    span = np.asarray(upper) - np.asarray(lower)
    velocities = rng.uniform(-0.1, 0.1, positions.shape) * span

    # 초기 위치를 평가하여 개인 및 전역 최적 위치 설정
    pbest = positions.copy()
    best_values = np.array([objective(x) for x in positions])
    gbest = pbest[np.argmin(best_values)].copy()

    for _ in range(iterations):
        # 관성항, 인지항, 사회항으로 속도 갱신
        r1 = rng.random(positions.shape)
        r2 = rng.random(positions.shape)
        inertia = 0.6 * velocities
        cognitive_term = 1.4 * r1 * (pbest - positions)
        social_term = 1.4 * r2 * (gbest - positions)
        velocities = inertia + cognitive_term + social_term

        # 위치 이동 후 경계값으로 제한, 벗어난 성분의 속도는 0으로 설정
        proposed = positions + velocities
        positions = np.clip(proposed, lower, upper)
        velocities[positions != proposed] = 0.0

        # 이동한 위치의 목적함수값 계산
        values = np.array([objective(x) for x in positions])

        # 기존 기록보다 좋은 위치로 개인 최적 위치 갱신
        improved = values < best_values
        pbest[improved] = positions[improved]
        best_values[improved] = values[improved]

        # 모든 개인 최적 위치 중 가장 좋은 위치 선택
        gbest = pbest[np.argmin(best_values)].copy()

    # 반복 종료 후 전역 최적 위치 반환
    return gbest


<style>
ul { padding-left:24px; }
table { margin-left:24px !important; margin-right:0 !important; }
th, td { text-align:left !important; }
li table { margin-left:0 !important; }
</style>

## 3) 예제

- 공통 PSO 함수에 생장량 표와 단계 변환·평가 함수를 전달, 20회 반복 후 조명·양액 단계와 생장량 출력


In [ ]:
# 조명 단계별(행), 양액 단계별(열) 생장량
growth = [[3, 5, 4], [6, 9, 7], [5, 8, 6]]

# 실수 위치를 가장 가까운 정수 단계로 변환
def decode(position):
    return np.floor(np.asarray(position) + 0.5).astype(int)

# 선택한 조명 및 양액 단계의 생장량 조회
def evaluate(decision):
    light, nutrient = decision
    return {"light": light, "nutrient": nutrient,
            "growth": growth[light][nutrient]}

# 생장량 최대화를 음수의 최소화로 변환
def objective(position):
    return -evaluate(decode(position))["growth"]

# 초기 후보 설정 및 PSO 실행
initial_positions = [[0, 0], [0, 2], [2, 0], [2, 2]]
result = particle_swarm(initial_positions, objective, 0, 2, iterations=20)

# 탐색 결과를 실제 의사결정으로 변환하고 출력
decision = decode(result)
print("Decision:", decision)
print("Growth (mm):", evaluate(decision)["growth"])
print("Stop reason: iteration limit")
